In [ ]:
import os
import pandas as pd
from lxml import etree

# 📂 Carpeta donde están los archivos XML
CARPETA_XML = "/Users/stephane.orts/Documents/Analysis/Electricity PVP/PVP 2024/pvpcdesglosehorario-2024"  # Cambia esto si los archivos están en otra carpeta
ARCHIVO_SALIDA = "datos_xml_limpios.csv"

# 📌 Función para obtener atributos XML de forma segura
def get_attrib(element, attrib):
    return element.get(attrib) if element is not None else None

# 📌 Lista para almacenar los datos de todos los archivos
todos_los_datos = []

# 📂 Recorrer todos los archivos XML en la carpeta
for archivo in os.listdir(CARPETA_XML):
    if archivo.endswith(".xml"):  # Procesar solo archivos XML
        ruta_xml = os.path.join(CARPETA_XML, archivo)
        print(f"📖 Procesando: {ruta_xml}")

        # Cargar el XML
        tree = etree.parse(ruta_xml)
        root = tree.getroot()

        # 📆 Extraer la fecha del nombre del archivo
        fecha_str = archivo[-12:-4]  # Extrae 'yyyymmdd'
        fecha_formateada = pd.to_datetime(fecha_str, format="%Y%m%d").strftime("%m/%d/%Y")

        # Extraer datos de SeriesTemporales
        series_data = []
        for serie in root.findall(".//{*}SeriesTemporales"):
            serie_dict = {
                "Fecha": fecha_formateada,  # Agregar la fecha extraída
                "TerminoCosteHorario": get_attrib(serie.find(".//{*}TerminoCosteHorario"), "v"),
                "TipoPrecio": get_attrib(serie.find(".//{*}TipoPrecio"), "v"),
            }
            
            # Extraer intervalos de tiempo dentro de SeriesTemporales
            for intervalo in serie.findall(".//{*}Intervalo"):
                serie_dict_copy = serie_dict.copy()  # Copia la serie base para cada intervalo
                serie_dict_copy["Hour"] = get_attrib(intervalo.find(".//{*}Pos"), "v")
                serie_dict_copy["Ctd"] = get_attrib(intervalo.find(".//{*}Ctd"), "v")
                series_data.append(serie_dict_copy)

        # Agregar los datos de este archivo a la lista general
        todos_los_datos.extend(series_data)

# 📌 Convertir a DataFrame
df = pd.DataFrame(todos_los_datos)

# 📌 1. Eliminar columnas innecesarias
df = df.drop(columns=["UnidadPrecio"], errors="ignore")  # Asegurar que existe antes de borrar

# 📌 2. Filtrar filas por criterios
df = df[(df["TerminoCosteHorario"] == "FEU") & (df["TipoPrecio"] == "Z14")]

# 📌 3. Reemplazar la columna 'Hour' (antes Pos) y cambiar 24 → 0
df["Hour"] = df["Hour"].replace(24, 0)

# 📌 4. Ordenar por Fecha y Hour
df = df.sort_values(by=["Fecha", "Hour"], ascending=[True, True])

# 📌 5. Guardar el archivo final limpio
df.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8")

# 📊 Mostrar las primeras filas del resultado
print(f"✅ Datos procesados y guardados en {ARCHIVO_SALIDA}")
print(df.head())

ModuleNotFoundError: No module named 'lxml'